# HW2-1 集成版本

**public 0.75869 / private 0.75778** —— 7 个模型概率加权平均。

本 notebook 假设 7 份 `probs.npy` 已经挂载到 `/kaggle/input`——用 kernel metadata 的 `kernel_sources` 直接引用成员 kernel 即可，不需要上传数据集：

```json
"kernel_sources": [
  "aviator22/ml2021spring-hw2-v7a", "aviator22/ml2021spring-hw2-v7b",
  "aviator22/ml2021spring-hw2-v8a", "aviator22/ml2021spring-hw2-v8b",
  "aviator22/ml2021spring-hw2-v5-mlp", "aviator22/ml2021spring-hw2-v4-cnn",
  "aviator22/ml2021spring-hw2-v3probs"
]
```

每份 `probs.npy` 的形状是 `(451552, 39)`、float16、已做过 TTA。

| 成员 | 架构 | 参数量 | 权重 | 从哪来 |
|---|---|---|---|---|
| v7a | MLP 1024/512/128 + ReLU | 2.80 M | 0.20 | `ml2021spring-hw2-v7a` |
| v7b | MLP 896/448/112 + ReLU | 2.38 M | 0.20 | `ml2021spring-hw2-v7b` |
| v8a | k=1 窗口 + ELU | 1.92 M | 0.18 | `ml2021spring-hw2-v8a` |
| v8b | 1280 宽 + GELU | 3.68 M | 0.18 | `ml2021spring-hw2-v8b` |
| v3 | MLP 1024/512/128（80% 数据，val 0.7495） | 2.80 M | 0.10 | `ml2021spring-hw2-v3probs` |
| v5 | 残差 1536 + GELU（过拟合，但误差互补） | 12.81 M | 0.08 | `ml2021spring-hw2-v5-mlp` |
| v4 | Conv1d over 55×39（最弱单体） | 0.31 M | 0.06 | `ml2021spring-hw2-v4-cnn` |

权重不是搜出来的（没有干净的 holdout 可用：v7 之后都用全量数据训练），
是按「单体强度 + 与主力的分歧度」手定的，然后一次性提交验证。改动权重的边际影响在 ±0.1 分内。


In [ ]:
import glob, os
import numpy as np
import pandas as pd

WEIGHTS = {'v7a': 0.20, 'v7b': 0.20, 'v8a': 0.18, 'v8b': 0.18, 'v3': 0.10, 'v5': 0.08, 'v4': 0.06}

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}_probs.npy', recursive=True) + \
           glob.glob(f'/kaggle/input/**/{name}/probs.npy', recursive=True)
    if not hits:
        raise FileNotFoundError(f'{name}_probs.npy 没挂上，见 README 的复现一节')
    return hits[0]

probs = {}
for k in WEIGHTS:
    probs[k] = np.load(find(k)).astype(np.float32)
    print(f'{k}: {probs[k].shape}  unique-argmax {np.unique(probs[k].argmax(1)).size}')


In [ ]:
base = probs['v7a'].argmax(1)
print('两两分歧率（v7a 为基准）:')
for k in probs:
    if k != 'v7a':
        print(f'  {k:4s} {(probs[k].argmax(1) != base).mean()*100:5.1f}%')
print('\n分歧率 11%~21% —— 这是集成有效的原因；单体分数只决定权重')


In [ ]:
mix = sum(WEIGHTS[k] * probs[k] for k in WEIGHTS)
pred = mix.argmax(1)
print('blended', pred.shape, '| classes used', np.unique(pred).size, '/ 39')

# 和主力的差异，用来确认集成确实改变了预测
print('与 v7a 单体分歧: %.1f%%' % ((pred != probs['v7a'].argmax(1)).mean()*100))


In [ ]:
SUB_PATH = glob.glob('/kaggle/input/**/sampleSubmission.csv', recursive=True)[0]
sub = pd.read_csv(SUB_PATH)
col = sub.columns[-1]
assert len(pred) == len(sub), (len(pred), len(sub))
sub[col] = pred
sub.to_csv('submission.csv', index=False)
print(sub.head()); print('rows', len(sub))
